In [14]:

# Install dependencies (run this cell first; restart kernel if your IDE recommends it after installs)
import sys
import subprocess

PACKAGES = [
    "pandas>=2.0.0",
    "numpy>=1.24.0",
    "nltk>=3.8.0",
    "gradio>=4.0.0",
]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("Installed:", ", ".join(PACKAGES))

Installed: pandas>=2.0.0, numpy>=1.24.0, nltk>=3.8.0, gradio>=4.0.0


# Section 1 — Probabilistic Spelling Correction (single notebook)

**Course:** CT052-3-M-NLP · **Corpus:** AG News **Business** (`label == 2`) · **Techniques:** minimum edit distance (Lab 07), bigram context (Lab 08), Bayesian-style ranking.

**Run order:** Execute cells from top to bottom. Ensure `data/ag_news_train.csv` exists (run `download_ag_news.py` once). The path cell finds the folder containing that file automatically.

In [15]:
from __future__ import annotations

import json
import math
import pickle
import re
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Imports OK.")

Imports OK.


## 1. Paths and configuration

`corpus/` will hold pickles, `sorted_words.txt`, and `corpus_metadata.json` for the report and GUI.

In [16]:
def find_project_root() -> Path:
    """Directory that contains data/ag_news_train.csv."""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data" / "ag_news_train.csv").is_file():
            return p
    raise FileNotFoundError(
        "Could not find data/ag_news_train.csv. Set working directory to the Spelling Correction folder."
    )

ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "ag_news_train.csv"
CORPUS_DIR = ROOT / "corpus"
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

BUSINESS_LABEL = 2
MIN_WORDS = 100_000
MAX_EDIT_DISTANCE = 2
LENGTH_WINDOW = 2
REAL_WORD_BIGRAM_THRESHOLD = 1e-4
NONWORD_SCORE_FLOOR = 1e-12
TOP_K = 5
MIN_TOKEN_LEN = 2

COMMON_SKIP = frozenset({
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "as", "by", "from", "with", "is", "are", "was", "were", "be",
    "been", "have", "has", "had", "it", "its", "this", "that", "these", "those",
})

print("ROOT:", ROOT)

ROOT: C:\Users\Phyo Phyo San\Documents\Master in APU\3 rd Sems\NLP\NLP Assignment - GP_V1


In [17]:
from pathlib import Path
print(Path.cwd())

c:\Users\Phyo Phyo San\Documents\Master in APU\3 rd Sems\NLP\NLP Assignment - GP_V1


## 2. Load AG News and keep Business articles only

Columns: `text`, `label`. We use `label == 2` (Business).

In [18]:
df = pd.read_csv(DATA_PATH)
biz = df[df["label"] == BUSINESS_LABEL].copy()
print("Business rows:", len(biz))
biz.head(2)

Business rows: 30000


,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2


## 3. Tokenize and build unigram + bigram counts

`nltk.word_tokenize` on lowercased text; edge punctuation stripped so types match the spelling check.

In [19]:
from nltk.tokenize import word_tokenize


def clean_token(raw: str) -> str | None:
    t = re.sub(r"^[^a-z0-9]+", "", raw.lower())
    t = re.sub(r"[^a-z0-9]+$", "", t)
    return t if t else None


def tokenize_document(text: str) -> list[str]:
    out = []
    for raw in word_tokenize(str(text).lower()):
        t = clean_token(raw)
        if t:
            out.append(t)
    return out


word_freq = Counter()
bigram_freq = Counter()

for text in biz["text"].astype(str):
    prev = None  # reset between articles (no cross-article bigram)
    for tok in tokenize_document(text):
        word_freq[tok] += 1
        if prev is not None:
            bigram_freq[(prev, tok)] += 1
        prev = tok

total_words = sum(word_freq.values())
vocabulary = set(word_freq.keys())

print("Total word tokens:", total_words)
print("Vocabulary size:", len(vocabulary))
print("Bigram types:", len(bigram_freq))
if total_words < MIN_WORDS:
    print("Note: token count below", MIN_WORDS, "- consider using more rows or all labels.")

Total word tokens: 1152747
Vocabulary size: 38354
Bigram types: 359288


## 4. Save corpus artifacts

Pickles for the corrector; `sorted_words.txt` for the dictionary panel; JSON metadata for the report.

In [20]:
metadata = {
    "dataset": "ag_news",
    "filter": "label==2 (Business)",
    "total_word_tokens": total_words,
    "vocabulary_size": len(vocabulary),
    "bigram_types": len(bigram_freq),
    "train_csv": str(DATA_PATH),
}

with open(CORPUS_DIR / "corpus_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

with open(CORPUS_DIR / "vocabulary.pkl", "wb") as f:
    pickle.dump(vocabulary, f)

with open(CORPUS_DIR / "word_frequencies.pkl", "wb") as f:
    pickle.dump(word_freq, f)

with open(CORPUS_DIR / "bigram_frequencies.pkl", "wb") as f:
    pickle.dump(bigram_freq, f)

with open(CORPUS_DIR / "sorted_words.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(sorted(vocabulary)))

print("Saved to", CORPUS_DIR)

Saved to C:\Users\Phyo Phyo San\Documents\Master in APU\3 rd Sems\NLP\NLP Assignment - GP_V1\corpus


## 5. Minimum edit distance (two-row DP, Lab 07)

Costs: match **0**, substitution **2**, insertion/deletion **1**.

In [21]:
def levenshtein_distance(s: str, t: str) -> int:
    M, N = len(s), len(t)
    if M == 0:
        return N
    if N == 0:
        return M
    if M > N:
        s, t = t, s
        M, N = N, M
    previous_row = list(range(M + 1))
    current_row = [0] * (M + 1)
    for j in range(1, N + 1):
        current_row[0] = j
        for i in range(1, M + 1):
            cost = 0 if s[i - 1] == t[j - 1] else 2
            current_row[i] = min(
                previous_row[i] + 1,
                current_row[i - 1] + 1,
                previous_row[i - 1] + cost,
            )
        previous_row, current_row = current_row, previous_row
    return previous_row[M]


print("kitten vs sitting ->", levenshtein_distance("kitten", "sitting"))

kitten vs sitting -> 5


## 6. `SpellingCorrector` — Bayesian-style ranking + bigrams

- **Non-word:** rank by `P(word) × exp(-2 × edit_distance)` (Lab 07 likelihood).
- **Real-word:** flag if `P(w | previous)` is below a threshold; suggest alternatives with higher bigram support and similar spelling.

In [ ]:
# =========================
# EDIT DISTANCE / ERROR MODEL
# =========================

COMMON_CONFUSIONS = {
    ('e', 'i'): 0.5,
    ('i', 'e'): 0.5,
    ('m', 'n'): 0.5,
    ('n', 'm'): 0.5,
    ('a', 's'): 0.5,
    ('s', 'a'): 0.5,
}

class SpellingCorrector:
    def __init__(
        self,
        vocabulary: set[str],
        word_freq: Counter,
        bigram_freq: Counter,
        total_words: int,
    ):
        self.vocabulary = vocabulary
        self.word_freq = word_freq
        self.bigram_freq = bigram_freq
        self.total_words = max(total_words, 1)

    def p_word(self, w: str) -> float:
        return self.word_freq[w] / self.total_words
    
    def p_bigram(self, w1: str, w2: str) -> float:
        c1 = self.word_freq[w1]
        V = len(self.vocabulary)

        # Laplace smoothing
        return (self.bigram_freq[(w1, w2)] + 1) / (c1 + V)

    def likelihood_ed(self, miss: str, cand: str) -> float:
        d = levenshtein_distance(miss, cand)
        base = math.exp(-2 * d)
        bonus = 1.0
        for a, b in zip(miss, cand):
            if (a, b) in COMMON_CONFUSIONS:
                bonus *= 1.5

        return base * bonus

    def candidate_words(self, typo: str) -> list[str]:
        L = len(typo)
        lo, hi = max(1, L - LENGTH_WINDOW), L + LENGTH_WINDOW
        return [w for w in self.vocabulary if lo <= len(w) <= hi]

    def suggest_nonword(self, typo: str) -> list[tuple[str, int, float]]:
        typo = clean_token(typo) or typo.lower()
        scores = []
        for w in self.candidate_words(typo):
            d = levenshtein_distance(typo, w)
            if d > MAX_EDIT_DISTANCE:
                continue
            prior = max(self.p_word(w), 1e-12)
            like = self.likelihood_ed(typo, w)
            post = prior * like
            if post < NONWORD_SCORE_FLOOR:
                continue
            scores.append((w, d, post))
        scores.sort(key=lambda x: x[2], reverse=True)
        return scores[:TOP_K]

    def suggest_for_realword(self, prev: str | None, wrong: str) -> list[tuple[str, int, float]]:
        wrong = clean_token(wrong) or wrong.lower()
        scores = []
        for w in self.candidate_words(wrong):
            d = levenshtein_distance(wrong, w)
            if d > MAX_EDIT_DISTANCE or w == wrong:
                continue
            prior = max(self.p_word(w), 1e-12)
            like = self.likelihood_ed(wrong, w)
            bg = 1.0
            if prev:
                bg = max(self.p_bigram(prev, w), 1e-12)
            post = prior * like * bg
            scores.append((w, d, post))
        scores.sort(key=lambda x: x[2], reverse=True)
        return scores[:TOP_K]

    def is_real_word_error(self, prev: str | None, tok: str) -> bool:
        t = clean_token(tok) or tok.lower()
        if t not in self.vocabulary or len(t) <= MIN_TOKEN_LEN or t in COMMON_SKIP:
            return False
        if not prev:
            return False
        prev_c = clean_token(prev) or prev.lower()
        if prev_c not in self.vocabulary:
            return False
        p = self.p_bigram(prev_c, t)
        return p < REAL_WORD_BIGRAM_THRESHOLD


corrector = SpellingCorrector(vocabulary, word_freq, bigram_freq, total_words)
print("Sample non-word suggestions for 'stck':", corrector.suggest_nonword("stck")[:3])

Sample non-word suggestions for 'stck': [('stock', 1, 0.00018467400091363653), ('stocks', 2, 4.2804128890597745e-05), ('stuck', 1, 4.343889404834426e-06)]


## 7. Error detection, annotated HTML, and replacement helpers

Token positions follow `word_tokenize`; flagged spans use wavy underlines in the HTML preview (red: non-word, blue: real-word). A mock checker is provided for deterministic UI demos.

In [23]:
def check_text(text: str, corr: SpellingCorrector) -> list[dict]:
    raw_tokens = word_tokenize(str(text))
    errors = []
    pos = 0
    prev_clean = None
    for raw in raw_tokens:
        start = text.find(raw, pos)
        if start < 0:
            start = pos
        end = start + len(raw)
        pos = end
        w = clean_token(raw)
        if w is None:
            continue
        if w not in corr.vocabulary:
            sug = corr.suggest_nonword(w)
            errors.append(
                {
                    "raw": raw,
                    "clean": w,
                    "start": start,
                    "end": end,
                    "kind": "non-word",
                    "suggestions": sug,
                    "p_bigram": None,
                }
            )
        elif corr.is_real_word_error(prev_clean, w):
            prev_c = clean_token(prev_clean) or prev_clean.lower()
            p_bg = corr.p_bigram(prev_c, w)
            sug = corr.suggest_for_realword(prev_clean, w)
            errors.append(
                {
                    "raw": raw,
                    "clean": w,
                    "start": start,
                    "end": end,
                    "kind": "real-word",
                    "suggestions": sug,
                    "p_bigram": p_bg,
                }
            )
        prev_clean = w
    return errors


def html_escape(s: str) -> str:
    return (
        s.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
    )


def highlight_underlined_html(text: str, errors: list) -> str:
    """Read-only preview with wavy underlines (non-word: red, real-word: blue)."""
    css = """
    <style>
    .sc-wrap { font-family: Inter, system-ui, sans-serif; font-size: 15px; line-height: 1.6;
      color: #e2e8f0; background: #1e293b; border: 1px solid #334155; border-radius: 0.75rem;
      padding: 12px 14px; min-height: 140px; }
    .sc-nw { text-decoration: underline wavy #f87171; text-underline-offset: 3px; cursor: pointer; }
    .sc-rw { text-decoration: underline wavy #60a5fa; text-underline-offset: 3px; cursor: pointer; }
    </style>
    """
    if not errors:
        body = html_escape(text).replace("\n", "<br>")
        return f"{css}<div class='sc-wrap'>{body}</div>"
    errors = sorted(errors, key=lambda e: e["start"])
    parts = []
    pos = 0
    for i, e in enumerate(errors):
        s, ed = e["start"], e["end"]
        cls = "sc-nw" if e["kind"] == "non-word" else "sc-rw"
        parts.append(html_escape(text[pos:s]))
        parts.append(
            f"<span class='{cls}' title='Error {i}'>{html_escape(text[s:ed])}</span>"
        )
        pos = ed
    parts.append(html_escape(text[pos:]))
    body = "".join(parts).replace("\n", "<br>")
    return f"{css}<div class='sc-wrap'>{body}</div>"


def replace_with_suggestion(text: str, errors: list, err_idx: int, sugg_idx: int) -> str:
    if err_idx is None or err_idx < 0 or err_idx >= len(errors):
        return text
    e = errors[err_idx]
    sug = e.get("suggestions") or []
    if sugg_idx < 0 or sugg_idx >= len(sug):
        return text
    new_word, _, _ = sug[sugg_idx]
    s, ed = e["start"], e["end"]
    return text[:s] + new_word + text[ed:]


def mock_check_spelling(text: str) -> list[dict]:
    """Deterministic stub for UI wiring; mirrors `check_text` structure."""
    t = str(text)
    demos = {
        "pateint": {
            "kind": "non-word",
            "suggestions": [("patient", 2, 2.55e-5), ("patent", 1, 1.41e-6)],
        },
        "cancr": {
            "kind": "non-word",
            "suggestions": [("cancer", 1, 3.2e-5), ("caner", 2, 1e-8)],
        },
        "diangosed": {
            "kind": "non-word",
            "suggestions": [("diagnosed", 1, 4.1e-5), ("diagnose", 2, 2e-7)],
        },
        "their": {
            "kind": "real-word",
            "p_bigram": 1e-12,
            "suggestions": [("the", 2, 9.52e-8), ("there", 1, 2.1e-7)],
        },
    }
    out: list[dict] = []
    for tok in word_tokenize(t):
        w = clean_token(tok)
        if not w or w.lower() not in demos:
            continue
        raw = tok
        start = t.find(raw)
        if start < 0:
            continue
        end = start + len(raw)
        spec = demos[w.lower()]
        err = {
            "raw": raw,
            "clean": w.lower(),
            "start": start,
            "end": end,
            "kind": spec["kind"],
            "suggestions": spec["suggestions"],
            "p_bigram": spec.get("p_bigram"),
        }
        out.append(err)
    return out


def format_correction_cards_html(errors: list, selected_idx: int | None) -> str:
    if not errors:
        return "<div style='color:#94a3b8;font-size:13px;padding:8px;'>No issues. Run Check spelling.</div>"
    blocks = []
    for i, e in enumerate(errors):
        kind = "Non-word" if e["kind"] == "non-word" else "Real-word (context)"
        border = "#38bdf8" if i == selected_idx else "#334155"
        bg = "#1e293b" if i == selected_idx else "#0f172a"
        pbg = ""
        if e.get("p_bigram") is not None:
            pbg = f"<div style='font-size:12px;color:#94a3b8;margin:6px 0;'>Bigram probability: {float(e['p_bigram']):.6e}</div>"
        sug_html = []
        for j, (w, d, sc) in enumerate(e.get("suggestions", [])[:TOP_K]):
            sug_html.append(
                f"<div style='display:flex;justify-content:space-between;align-items:center;padding:6px 8px;margin:4px 0;"
                f"background:#1e293b;border:1px solid #334155;border-radius:8px;'>"
                f"<span style='color:#e2e8f0;font-weight:500;'>✓ {html_escape(w)}</span>"
                f"<span style='font-size:11px;color:#94a3b8;'>Dist: {d} | Score: {sc:.2e}</span></div>"
            )
        blocks.append(
            f"<div style='border:1px solid {border};border-radius:12px;padding:12px;margin-bottom:10px;background:{bg};"
            f"box-shadow:0 1px 3px rgba(0,0,0,0.25);'>"
            f"<div style='font-size:12px;color:#94a3b8;'>Flagged</div>"
            f"<div style='font-size:16px;font-weight:600;color:#f8fafc;margin:4px 0;'>{html_escape(e['raw'])}</div>"
            f"<div style='font-size:13px;color:#38bdf8;margin-bottom:6px;'>{kind}</div>"
            f"{pbg}"
            f"<div style='font-size:12px;color:#94a3b8;margin-top:8px;'>Top suggestions</div>"
            f"{''.join(sug_html)}"
            f"</div>"
        )
    return "<div style='max-height:min(70vh,520px);overflow-y:auto;padding-right:4px;'>" + "".join(blocks) + "</div>"


_sample = "The stk market rose sharply acording to analysts."
print("Real check:", len(check_text(_sample, corrector)), "issues")

Real check: 2 issues


## 8. Web interface (Gradio)

A tabbed layout provides the spell-checking workflow and a corpus browser. The interface uses a cool-toned theme, collapsible information panels, an annotated HTML preview with wavy underlines, and assistant cards with suggestion controls. Relaunching `demo.launch` in the same kernel may require a kernel restart.

In [27]:
import gradio as gr
import pandas as pd

MAX_CHARS = 500

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
.gradio-container, .gradio-container * { font-family: Inter, system-ui, sans-serif !important; }
.gradio-container { background: #0f172a !important; color: #e2e8f8 !important; }
footer { display: none !important; }
"""

DOMAIN_LABEL = "Business / Finance (AG News Business headlines)"


def _ser_errors(errs: list) -> list:
    out = []
    for e in errs:
        ee = {k: v for k, v in e.items() if k != "suggestions"}
        ee["suggestions"] = [list(t) for t in e.get("suggestions", [])]
        out.append(ee)
    return out


def _deser_errors(errs: list) -> list:
    out = []
    for e in errs:
        ee = dict(e)
        ee["suggestions"] = [tuple(x) for x in ee.get("suggestions", [])]
        out.append(ee)
    return out


def _sugg_button_updates(err: dict | None) -> tuple:
    if not err:
        return tuple(gr.update(visible=False) for _ in range(5))
    sug = err.get("suggestions", [])
    upd = []
    for i in range(5):
        if i < len(sug):
            w, d, sc = sug[i]
            upd.append(
                gr.update(
                    visible=True,
                    value=f"✓ {w}  ·  Dist {d}  ·  Score {sc:.2e}",
                )
            )
        else:
            upd.append(gr.update(visible=False))
    return tuple(upd)


def corpus_table(query: str) -> pd.DataFrame:
    q = (query or "").strip().lower()
    items = list(word_freq.items())
    if q:
        items = [(w, c) for w, c in items if q in w.lower()]
    items.sort(key=lambda x: (-x[1], x[0]))
    return pd.DataFrame(items[:8000], columns=["Word", "Frequency"])


def run_check(editor_text: str, use_mock: bool):
    t = (editor_text or "")[:MAX_CHARS]
    if not t.strip():
        z = [gr.update(visible=False)] * 5
        return (
            highlight_underlined_html("", []),
            format_correction_cards_html([], None),
            gr.update(choices=[], value=None),
            [],
            f"0 / {MAX_CHARS}",
            f"Issues: **0** · Characters: **0**",
            *z,
        )
    errs = mock_check_spelling(t) if use_mock else check_text(t, corrector)
    ser = _ser_errors(errs)
    choices = [(f"{e['raw']} · {e['kind']}", str(i)) for i, e in enumerate(errs)]
    preview = highlight_underlined_html(t, errs)
    if not errs:
        z = [gr.update(visible=False)] * 5
        return (
            preview,
            format_correction_cards_html([], None),
            gr.update(choices=[], value=None),
            ser,
            f"{len(t)} / {MAX_CHARS}",
            f"Issues: **0** · Characters: **{len(t)}**",
            *z,
        )
    cards = format_correction_cards_html(errs, 0)
    btns = _sugg_button_updates(errs[0])
    status = f"Issues: **{len(errs)}** · Characters: **{len(t)}**"
    return (
        preview,
        cards,
        gr.update(choices=choices, value="0"),
        ser,
        f"{len(t)} / {MAX_CHARS}",
        status,
        *btns,
    )


def on_pick_error(err_key: str | None, ser: list):
    errs = _deser_errors(ser or [])
    if not errs or err_key is None:
        return (format_correction_cards_html(errs, None), *[gr.update(visible=False)] * 5)
    try:
        idx = int(err_key)
    except (TypeError, ValueError):
        return (format_correction_cards_html(errs, None), *[gr.update(visible=False)] * 5)
    cards = format_correction_cards_html(errs, idx)
    btns = _sugg_button_updates(errs[idx])
    return (cards, *btns)


def replace_suggestion_click(editor_text: str, ser: list, err_key: str | None, use_mock: bool, sugg_idx: int):
    errs = _deser_errors(ser or [])
    if err_key is None or not errs:
        z = [gr.update(visible=False)] * 5
        return (
            editor_text or "",
            [],
            highlight_underlined_html(editor_text or "", []),
            format_correction_cards_html([], None),
            gr.update(choices=[], value=None),
            f"{len(editor_text or '')} / {MAX_CHARS}",
            f"Issues: **0**",
            *z,
        )
    ei = int(err_key)
    new_t = replace_with_suggestion((editor_text or "")[:MAX_CHARS], errs, ei, sugg_idx)
    new_t = new_t[:MAX_CHARS]
    new_errs = mock_check_spelling(new_t) if use_mock else check_text(new_t, corrector)
    ser2 = _ser_errors(new_errs)
    preview = highlight_underlined_html(new_t, new_errs)
    if not new_errs:
        z = [gr.update(visible=False)] * 5
        return (
            new_t,
            ser2,
            preview,
            format_correction_cards_html([], None),
            gr.update(choices=[], value=None),
            f"{len(new_t)} / {MAX_CHARS}",
            f"Issues: **0** · Characters: **{len(new_t)}**",
            *z,
        )
    choices = [(f"{e['raw']} · {e['kind']}", str(i)) for i, e in enumerate(new_errs)]
    cards = format_correction_cards_html(new_errs, 0)
    btns = _sugg_button_updates(new_errs[0])
    status = f"Issues: **{len(new_errs)}** · Characters: **{len(new_t)}**"
    return (
        new_t,
        ser2,
        preview,
        cards,
        gr.update(choices=choices, value="0"),
        f"{len(new_t)} / {MAX_CHARS}",
        status,
        *btns,
    )


INFO_MD = """
**Techniques:** Levenshtein (minimum edit distance), bigram probabilities, Bayesian-style candidate scoring.

**Error types:** Non-word (out-of-vocabulary) and real-word (low contextual bigram probability).
"""

USAGE_MD = """
1. Enter up to 500 characters.  
2. Optionally enable **Mock demo** for fixed sample typos (`pateint`, `cancr`, `diangosed`, `their`).  
3. Click **Check spelling**.  
4. Select a flagged word in the assistant to focus its card.  
5. Use **S1–S5** to apply the listed suggestion.  
6. Open **Corpus Browser** to search vocabulary frequencies.
"""


with gr.Blocks(title="Spelling correction — Section 1", css=CUSTOM_CSS) as demo:
    err_state = gr.State([])

    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=240):
            with gr.Accordion("System information", open=False):
                gr.Markdown(INFO_MD)
            with gr.Accordion("Usage", open=False):
                gr.Markdown(USAGE_MD)

        with gr.Column(scale=5):
            gr.Markdown("## Spelling correction system")
            with gr.Tabs():
                with gr.Tab("Spell Checker"):
                    with gr.Row():
                        with gr.Column(scale=7):
                            use_mock = gr.Checkbox(
                                label="Mock demo (deterministic sample errors)",
                                value=False,
                            )
                            editor = gr.Textbox(
                                label="Editor (500 characters)",
                                lines=12,
                                max_lines=16,
                                max_length=MAX_CHARS,
                            )
                            preview = gr.HTML(label="Annotated preview")
                            with gr.Row():
                                gr.Markdown("")
                                char_md = gr.Markdown(f"0 / {MAX_CHARS}")
                            check_btn = gr.Button("Check spelling", variant="primary")
                            status_md = gr.Markdown("Issues: **0**")

                        with gr.Column(scale=3):
                            gr.Markdown("### Assistant")
                            assistant = gr.HTML()
                            err_pick = gr.Dropdown(
                                label="Select flagged word",
                                choices=[],
                                interactive=True,
                            )
                            gr.Markdown("**Suggestions**")
                            sugg_btns = [gr.Button(f"S{i + 1}", visible=False, size="sm") for i in range(5)]

                with gr.Tab("Corpus Browser"):
                    gr.Markdown(
                        f"**Total word tokens:** {total_words:,} · **Unique words:** {len(vocabulary):,} · **Domain:** {DOMAIN_LABEL}"
                    )
                    corp_q = gr.Textbox(label="Search words", placeholder="Substring filter…")
                    corp_df = gr.Dataframe(
                        label="Vocabulary",
                        value=corpus_table(""),
                        interactive=False,
                        wrap=True,
                    )

    corp_q.change(corpus_table, corp_q, corp_df)

    check_outputs = [preview, assistant, err_pick, err_state, char_md, status_md, *sugg_btns]
    check_btn.click(run_check, [editor, use_mock], check_outputs)

    err_pick.change(on_pick_error, [err_pick, err_state], [assistant, *sugg_btns])

    for i in range(5):
        sugg_btns[i].click(
            lambda et, st, ek, um, ix=i: replace_suggestion_click(et, st, ek, um, ix),
            [editor, err_state, err_pick, use_mock],
            [editor, err_state, preview, assistant, err_pick, char_md, status_md, *sugg_btns],
        )

    editor.change(lambda t: f"{len((t or '')[:MAX_CHARS])} / {MAX_CHARS}", editor, char_md)

demo.launch(share=False, inline=True)



C:\Users\Phyo Phyo San\AppData\Local\Temp\ipykernel_74300\1460490710.py:181: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Spelling correction — Section 1", css=CUSTOM_CSS) as demo:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
